# Visualizing Average Latents

This notebook visualizes the average latents created in `proto_note.ipynb`

In [1]:
# setup libraries and functions

import torch
if torch.cuda.is_available():
    print("GPU is available.")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.current_device()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}") # Assuming at least one GPU
else:
    print("GPU is not available.")

import sys
!{sys.executable} -m pip install av

from diffusers import AutoencoderKLCogVideoX
from diffusers.utils import export_to_video
import gc
import numpy as np

import torchvision
from torchvision import transforms

torch.cuda.empty_cache()
gc.collect()

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# load 3d vae
# use float16 to save memory
model_id = "THUDM/CogVideoX-2b"
vae = AutoencoderKLCogVideoX.from_pretrained(
    model_id,
    subfolder='vae',
    torch_dtype=torch.float16,
).to(device)

# enables tiling which should save on vram usage
vae.enable_tiling()

# helper to prep video for vae
def load_and_process_video(video_path, height=480, width=720, max_frames=16):
    print(f"Loading {video_path}...")

    # read video
    video_frames, _, _ = torchvision.io.read_video(video_path, output_format="TCHW", pts_unit='sec')

    # cap frames
    if len(video_frames) > max_frames:
        video_frames = video_frames[:max_frames]

    current_frames = len(video_frames)

    # transform pipeline
    transform = transforms.Compose([
        transforms.Resize(height, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.CenterCrop((height, width)),
    ])

    # apply transforms
    processed_frames = torch.stack([transform(f) for f in video_frames])

    # format for VAE
    video_tensor = processed_frames.permute(1, 0, 2, 3).unsqueeze(0)

    # normalize
    video_tensor = video_tensor.float() / 255.0  # Now [0, 1]
    video_tensor = (video_tensor * 2.0) - 1.0    # Now [-1, 1]

    return video_tensor.to(device, dtype=torch.float16)

# manually splits vid into temporal chunks to save vram
def get_latents_chunked(video_tensor, chunk_size=4):
    frames = video_tensor.shape[2]
    latent_list = []

    with torch.no_grad():
        for i in range(0, frames, chunk_size):
            # get slice of frame
            end = min(i + chunk_size, frames)
            video_chunk = video_tensor[:,:,i:end,:,:]

            # encode current chunk
            posterior = vae.encode(video_chunk).latent_dist
            latents = posterior.sample() * vae.config.scaling_factor
            latent_list.append(latents)

            # clean up vram
            del video_chunk, posterior, latents
            torch.cuda.empty_cache()

    # stitch chunks together and return
    return torch.cat(latent_list, dim=2)


# decode latents & manually split to save vram
def decode_latents_chunked(latents, chunk_size=1):
    latent_frames_count = latents.shape[2]
    decoded_video_list = []

    with torch.no_grad():
        for i in range(0, latent_frames_count, chunk_size):
            # slice latent
            end = min(i+chunk_size, latent_frames_count)
            latent_chunk = latents[:,:,i:end,:,:]

            # decode current chunk
            frames = vae.decode(latent_chunk).sample

            # move to cpu to free vram
            frames = (frames / 2 + 0.5).clamp(0,1)
            decoded_video_list.append(frames.cpu())

            # cleanup
            del latent_chunk, frames
            torch.cuda.empty_cache()
    # stitch together and return
    return torch.cat(decoded_video_list, dim=2)


GPU is available.
Number of GPUs: 1
Current GPU: 0
GPU name: Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 22.9 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/862M [00:00<?, ?B/s]

In [3]:

# Load the .pt file
class_average_latents = torch.load('/content/class_average_latents.pt')
classes = list(class_average_latents.keys())

print(f"Loaded {len(classes)} classes")
print(classes)

Loaded 13 classes
['yoga', 'texting', 'welding', 'bartending', 'zumba', 'laughing', 'cartwheeling', 'motorcycling', 'archery', 'sailing', 'dodgeball', 'jogging', 'headbanging']


# Decode Latents & Display Videos
Load the `class_average_latents.pt` file into content and choose from any of the classes to visualize their average.

In [12]:
from IPython.display import HTML
from base64 import b64encode
OUTPUT_FILE = 'output.mp4'

# change class to visualize here
class_name = classes[9]

vid_latent = class_average_latents[class_name]

decoded_frames = decode_latents_chunked(vid_latent)

video_tensor = decoded_frames[0]
video_tensor = video_tensor.permute(1,2,3,0)
video_np = video_tensor.cpu().numpy()
video_np = (video_np * 255).astype(np.uint8)

export_to_video(video_np, 'output.mp4', fps=16)

mp4 = open(OUTPUT_FILE,'rb').read()
print(f'Latent Average of {class_name}')
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=400 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

Latent Average of sailing
